# Normalización y regularización en CNN con PyTorch

Compararemos Batch Normalization, Cutout, Mixup y suavizado de etiquetas en una CNN pequeña para CIFAR-10. Cada variante usa la misma partición.

## Entorno y datos

Ejecuta con el kernel **IPD434**. PyTorch utiliza CUDA si está disponible. CIFAR-10 se descarga la primera vez en `../data/cifar10`. Usamos subconjuntos para que el ejemplo sea corto.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| dispositivo:', device)

In [ ]:
DATA_ROOT = Path('../data/cifar10')
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
dataset = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=transform)
test_full = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=transform)
generator = torch.Generator().manual_seed(SEED)
train_set, val_set, unused = random_split(dataset, [3000, 1000, 46000], generator=generator)
test_set, unused_test = random_split(test_full, [1000, 9000], generator=torch.Generator().manual_seed(SEED + 1))
train_loader = DataLoader(train_set, batch_size=128, shuffle=True, generator=torch.Generator().manual_seed(SEED), num_workers=0)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False, num_workers=0)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=0)
print('Particiones:', len(train_set), len(val_set), len(test_set))

## Cutout y Mixup

Cutout oculta un parche de la imagen. Mixup combina dos imágenes y pondera la pérdida con las etiquetas de ambas. En validación no se alteran los datos.

In [ ]:
def apply_cutout(images, patch_size=8):
    result = images.clone()
    _, _, height, width = result.shape
    for i in range(result.size(0)):
        center_y = int(torch.randint(height, (1,)).item())
        center_x = int(torch.randint(width, (1,)).item())
        y1, y2 = max(0, center_y - patch_size // 2), min(height, center_y + patch_size // 2)
        x1, x2 = max(0, center_x - patch_size // 2), min(width, center_x + patch_size // 2)
        result[i, :, y1:y2, x1:x2] = 0.0
    return result

def apply_mixup(images, labels, alpha=0.2):
    lam = float(np.random.beta(alpha, alpha))
    permutation = torch.randperm(images.size(0), device=images.device)
    mixed = lam * images + (1 - lam) * images[permutation]
    return mixed, labels, labels[permutation], lam

example_images, example_labels = next(iter(train_loader))
cutout_example = apply_cutout(example_images[:1], patch_size=10)[0]
mixup_example, _, _, mix_lambda = apply_mixup(example_images[:2], example_labels[:2])
mean_tensor = torch.tensor(mean).view(3, 1, 1)
std_tensor = torch.tensor(std).view(3, 1, 1)
fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, image, title in zip(axes, [example_images[0], cutout_example, mixup_example[0]], ['Original', 'Cutout', f'Mixup λ={mix_lambda:.2f}']):
    ax.imshow((image.cpu() * std_tensor + mean_tensor).permute(1, 2, 0).clamp(0, 1))
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## CNN y ciclo de entrenamiento

Batch Normalization normaliza activaciones internas. Label smoothing evita que la pérdida exija probabilidad 1 para la clase correcta. Usaremos la misma CNN y el mismo optimizador para comparar estas opciones y las transformaciones.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, batch_norm=False):
        super().__init__()
        def conv_block(in_channels, out_channels):
            block = [nn.Conv2d(in_channels, out_channels, 3, padding=1)]
            if batch_norm:
                block.append(nn.BatchNorm2d(out_channels))
            block += [nn.ReLU(), nn.MaxPool2d(2)]
            return block
        self.features = nn.Sequential(
            *conv_block(3, 32), *conv_block(32, 64),
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.Linear(64, 10),
        )
    def forward(self, images):
        return self.features(images)

def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += criterion(logits, labels).item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            total += len(labels)
    return total_loss / total, correct / total

def train_variant(name, batch_norm=False, augmentation=None, label_smoothing=0.0, epochs=4):
    torch.manual_seed(SEED)
    model = SmallCNN(batch_norm=batch_norm).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    history = {'train_loss': [], 'val_loss': [], 'val_accuracy': []}
    for epoch in range(1, epochs + 1):
        model.train()
        train_sum, train_count = 0.0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            if augmentation == 'cutout':
                images = apply_cutout(images)
            optimizer.zero_grad(set_to_none=True)
            if augmentation == 'mixup':
                mixed, labels_a, labels_b, lam = apply_mixup(images, labels)
                logits = model(mixed)
                loss = lam * criterion(logits, labels_a) + (1 - lam) * criterion(logits, labels_b)
            else:
                logits = model(images)
                loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_sum += loss.item() * len(labels)
            train_count += len(labels)
        train_loss = train_sum / train_count
        val_loss, val_accuracy = evaluate(model, val_loader)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_accuracy)
        print(f'{name} | época {epoch}/{epochs} | val_loss={val_loss:.4f} | val_acc={val_accuracy:.3f}')
    return model, history

## Comparar estrategias

Cada variante usa el mismo número de épocas. La validación sirve para comparar y la prueba se reserva para la configuración elegida.

In [ ]:
variants = {
    'Base': {'batch_norm': False},
    'BatchNorm': {'batch_norm': True},
    'Cutout': {'augmentation': 'cutout'},
    'Mixup': {'augmentation': 'mixup'},
    'Label smoothing': {'label_smoothing': 0.1},
}
models_by_name, histories, rows = {}, {}, []
for name, options in variants.items():
    model, history = train_variant(name, **options, epochs=4)
    models_by_name[name], histories[name] = model, history
    rows.append({
        'estrategia': name,
        'última pérdida entrenamiento': history['train_loss'][-1],
        'última pérdida validación': history['val_loss'][-1],
        'última exactitud validación': history['val_accuracy'][-1],
    })
summary = pd.DataFrame(rows).sort_values('última pérdida validación')
display(summary.reset_index(drop=True))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in histories.items():
    axes[0].plot(history['train_loss'], label=name)
    axes[1].plot(history['val_loss'], label=name)
axes[0].set(title='Pérdida de entrenamiento', xlabel='Época', ylabel='Cross entropy')
axes[1].set(title='Pérdida de validación', xlabel='Época', ylabel='Cross entropy')
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Evaluación final

Elegimos la estrategia por pérdida de validación y evaluamos una vez en el conjunto de prueba. Los resultados de esta ejecución breve son ilustrativos.

In [ ]:
best_name = summary.iloc[0]['estrategia']
test_loss, test_accuracy = evaluate(models_by_name[best_name], test_loader)
print('Estrategia elegida:', best_name)
print(f'Prueba | pérdida={test_loss:.4f} | exactitud={test_accuracy:.3f}')